See more details in sqlite-ingest.ipynb

In [1]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

In [15]:
# connect to DB and see how many docs are in the index
sqlite_index.count()

85

In [16]:
results = sqlite_index.search("Can I still join the course after it started?", num_results=5)
[doc["question"] for doc in results]

['I just discovered the course. Can I still join?',
 'How do I start using Google Gemini models in the Module 1 notebook through the OpenAI-compatible endpoint?',
 'I missed the first homework - can I still get a certificate?',
 'Homework: Why does the content keep changing?',
 'What happens to code saved in Codespaces if I do not commit it?']

## Use RAGBase with this new search index

In [17]:
from rag_helper import RAGBase
from openai import OpenAI
from dotenv import load_dotenv

openai_client = OpenAI()
load_dotenv()

assistant = RAGBase(
    index=sqlite_index, # only thing to swap
    llm_client=openai_client,
)

In [18]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join after it started. If you want a certificate, make sure to submit your project while submissions are still open.


Swapping the index was easy. We can see SQLLite search index gives us a similar result to minsearch index. 

If the API for sqlitesearch was different than minsearch, we need to: 

1. subclass RAGBase
2. override the build_index method

In [19]:
# close DB connection when done
sqlite_index.close()